In [ ]:
pip install xgboost

In [ ]:
!pip install imbalanced-learn

In [ ]:
import numpy as np
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_score,
                             recall_score, f1_score)

def evaluate(y_true, proba):
    y_true = np.asarray(y_true)
    prec, rec, thr = precision_recall_curve(y_true, proba)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    best_t = float(thr[np.argmax(f1[:-1])])
    y_pred = (proba >= best_t).astype(int)
    
    return {
        'Macro_AUC':   roc_auc_score(y_true, proba),          
        'Precision+':  precision_score(y_true, y_pred, pos_label=1),
        'Recall+':     recall_score(y_true, y_pred, pos_label=1),
        'F1+':         f1_score(y_true, y_pred, pos_label=1),
        'Macro_F1':    f1_score(y_true, y_pred, average='macro'),
        'ROC_AUC':     roc_auc_score(y_true, proba),
        'PR_AUC':      average_precision_score(y_true, proba),
        'threshold':   best_t,
    }

In [ ]:
import pandas as pd

BASE = '/home/jupyter/workspace/rw-migration-aou-rw-24b38658'
MATRIX_DIR = f'{BASE}/amia/new_survey'

matrices = {}
for k in ['ehrdemo_6', 'ehrdemo_survnobasic_6',
          'ehrdemo_12', 'ehrdemo_survnobasic_12',
          'ehrdemo_24', 'ehrdemo_survnobasic_24']:
    matrices[k] = pd.read_parquet(f'{MATRIX_DIR}/matrix_{k}.parquet')
    print(f"✓ {k}: {matrices[k].shape}")

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
TEST_SIZE = 0.20

BASE_PARAMS = dict(
    objective="binary:logistic", eval_metric=["auc", "aucpr"],   
    eta=0.03, max_depth=5, min_child_weight=16,
    subsample=0.75, colsample_bytree=0.70,
    reg_lambda=3.0, reg_alpha=0.4, tree_method="hist",
)

df = matrices['ehrdemo_survnobasic_24']
y = df['IsPositive']
X = df.drop(columns=['person_id', 'IsPositive'])

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=RANDOM_STATE
)

neg, pos = (y_tr == 0).sum(), (y_tr == 1).sum()
base_spw = float(neg / max(pos, 1))
print(f"Train: pos={pos:,} neg={neg:,} | base_spw={base_spw:.2f}\n")

dva = xgb.DMatrix(X_va, label=y_va)   

def run(name, X_train, y_train, spw):
    params = BASE_PARAMS.copy()
    params["scale_pos_weight"] = spw
    dtr = xgb.DMatrix(X_train, label=y_train)
    bst = xgb.train(params, dtr, num_boost_round=5000,
                    evals=[(dva, "valid")], early_stopping_rounds=200,
                    verbose_eval=False)
    proba = bst.predict(dva, iteration_range=(0, bst.best_iteration + 1))
    m = evaluate(y_va, proba)                                   
    m = {'method': name, 'train_n': len(y_train), **m}
    print(f"[{name:<22}] PR-AUC={m['PR_AUC']:.4f} | ROC-AUC={m['ROC_AUC']:.4f} | "
          f"F1+={m['F1+']:.4f} | P+={m['Precision+']:.4f} | R+={m['Recall+']:.4f}")
    return m

results = []
results.append(run("A. scale_pos_weight", X_tr, y_tr, base_spw * 0.5))
results.append(run("B. no adjustment",    X_tr, y_tr, 1.0))

print("\n[SMOTE] sampling_strategy=0.33 ...")
X_tr_filled = X_tr.fillna(0).astype('float32')
X_sm, y_sm = SMOTE(sampling_strategy=0.33, random_state=RANDOM_STATE).fit_resample(X_tr_filled, y_tr)
print(f"  pos={int((y_sm==1).sum()):,} neg={int((y_sm==0).sum()):,}")
del X_tr_filled
results.append(run("C. SMOTE", X_sm, y_sm, 1.0))
del X_sm, y_sm

print("\n[RandomUnderSampler] ...")
X_rus, y_rus = RandomUnderSampler(random_state=RANDOM_STATE).fit_resample(X_tr, y_tr)
print(f"  pos={int((y_rus==1).sum()):,} neg={int((y_rus==0).sum()):,}")
results.append(run("D. RandomUnderSampler", X_rus, y_rus, 1.0))

print(f"\n{'='*60}\nIMBALANCE COMPARISON (ehr_surv_24m, validation set)\n{'='*60}")
cols = ['method','train_n','PR_AUC','ROC_AUC','F1+','Precision+','Recall+','Macro_F1']
print(pd.DataFrame(results)[cols].to_string(index=False))